# mama — Google Colab Export

This notebook was generated by mama. It reproduces your training run in Google Colab.

## Setup
1. Upload your training output folder (the one containing `training_config.json`) to this Colab runtime.
2. Or upload the `training_config.json` file directly and update the path below.
3. Run all cells (Runtime → Run all).

In [ ]:
# --- CONFIG ---
# Update this to the path where you uploaded your training_config.json
CONFIG_PATH = "/content/training_config.json"

## 1. Install Dependencies

In [ ]:
import subprocess, sys, json, os
from pathlib import Path

def run(cmd):
    print(f"$ {cmd}")
    subprocess.check_call(cmd, shell=True)

run("pip install -q torch transformers datasets accelerate tensorboard xformers")

## 2. Load Training Config

In [ ]:
config_path = Path(CONFIG_PATH)
if not config_path.exists():
    raise FileNotFoundError(f"training_config.json not found at {CONFIG_PATH}. Upload your config file and update CONFIG_PATH above.")

with open(config_path) as f:
    cfg = json.load(f)

print("Loaded config:")
print(json.dumps(cfg, indent=2))

model_name = cfg.get("model_name_or_path", "")
dataset_path = cfg.get("dataset_path", "")
output_dir = Path(cfg.get("output_dir", "/content/outputs"))
output_dir.mkdir(parents=True, exist_ok=True)

## 3. Load Model & Tokenizer

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

print(f"Loading model: {model_name}")
tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

use_qlora = cfg.get("use_qlora", False)
use_lora = cfg.get("use_lora", False)

load_kwargs = {"torch_dtype": "auto"}
model = AutoModelForCausalLM.from_pretrained(model_name, **load_kwargs)

## 4. Load & Tokenize Dataset

In [ ]:
import datasets

print(f"Loading dataset: {dataset_path}")

path_obj = Path(dataset_path)
if path_obj.exists():
    if path_obj.is_dir():
        dataset = datasets.load_from_disk(str(path_obj))
    else:
        dataset = datasets.load_dataset(
            "json" if path_obj.suffix == ".jsonl" else "csv",
            data_files=str(path_obj),
            split="train",
        )
else:
    dataset = datasets.load_dataset(dataset_path, split="train")

text_column = cfg.get("text_column", "text")
max_samples = cfg.get("max_samples", None)
if max_samples:
    dataset = dataset.select(range(min(max_samples, len(dataset))))

def tokenize_function(examples):
    return tokenizer(
        examples[text_column],
        padding="max_length",
        truncation=True,
        max_length=cfg.get("max_seq_length", 2048),
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True)
print(f"Dataset ready: {len(tokenized_dataset)} samples")

## 5. Configure & Run Training

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

training_args = TrainingArguments(
    output_dir=str(output_dir),
    overwrite_output_dir=True,
    per_device_train_batch_size=cfg.get("per_device_train_batch_size", 2),
    gradient_accumulation_steps=cfg.get("gradient_accumulation_steps", 4),
    learning_rate=cfg.get("learning_rate", 2e-4),
    num_train_epochs=cfg.get("num_train_epochs", 3),
    max_steps=cfg.get("max_steps", -1),
    warmup_steps=cfg.get("warmup_steps", 100),
    logging_steps=cfg.get("logging_steps", 10),
    save_steps=cfg.get("save_steps", 500),
    lr_scheduler_type=cfg.get("lr_scheduler_type", "cosine"),
    gradient_checkpointing=cfg.get("gradient_checkpointing", False),
    bf16=cfg.get("bf16", False),
    logging_dir=str(output_dir / "logs"),
    report_to="tensorboard",
    ddp_find_unused_parameters=False if cfg.get("use_lora", False) else None,
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

print("Starting training...")
trainer.train()

## 6. Save Model

In [ ]:
final_dir = output_dir / "final_model"
trainer.save_model(str(final_dir))
tokenizer.save_pretrained(str(final_dir))
print(f"Model saved to {final_dir}")
print("\nDone! Download your outputs from the Files panel on the left.")